<a href="https://colab.research.google.com/github/sethmwhite/worth-it/blob/main/Worth_It.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WorthIt
My project helps gamers on a budget decide which Steam games to buy this week by matching highly rated games to the genre, budget, and operating system they choose, as well as checking if a game is on sale.

In [31]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import requests
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 1000 games from: /content/drive/MyDrive/MIST4600/


In [32]:
#Return the games in a genre that cost no more than budget, best rated first.

#Set include_free=False, or change with_free to no_free (in the print) to leave out free games.


def fit_games(games, genre, budget, include_free=True):
    in_genre = games[games["genres"].str.contains(genre, case=False, na=False)]
    affordable = in_genre[in_genre["price"] <= budget]
    if include_free == False:
        affordable = affordable[affordable["price"] > 0]
    return affordable.sort_values("pct_pos_total", ascending=False)

with_free = fit_games(games, "Action", 30, include_free=True)
no_free = fit_games(games, "Action", 30, include_free=False)

print(with_free[["name", "price", "pct_pos_total"]].head(10))

                          name  price  pct_pos_total
473  HoloCure - Save the Fans!   0.00             99
63           Vampire Survivors   4.99             98
57                       Hades  24.99             98
54           People Playground   9.99             98
311                Katana ZERO  14.99             98
920                    VTOL VR  29.99             98
320                Pizza Tower  19.99             98
893              Super Hexagon   2.99             97
115                 Dead Cells  12.49             97
605          The Wolf Among Us  14.99             97


In [33]:
#Return the games that run on the given os_name ('windows', 'mac', or 'linux').

#prints the number of "fitting" games based on your search, as well as the number of those games that run on your OS.

def games_for_os(games, os_name):
    return games[games[os_name] == True]

fit = fit_games(games, "RPG", 60, include_free=False)
runs_on_mac = games_for_os(fit, "mac")

print(f"Matching Games: {len(fit)}")
print(f"OS-Compatible Games: {len(runs_on_mac)}")
print(runs_on_mac[["name", "price", "pct_pos_total"]].head(15))

Matching Games: 209
OS-Compatible Games: 69
                                     name  price  pct_pos_total
13                         Stardew Valley  14.99             98
57                                  Hades  24.99             98
63                      Vampire Survivors   4.99             98
712               Touhou Mystia's Izakaya   5.99             98
125                Mount & Blade: Warband  19.99             97
148                        DAVE THE DIVER  19.99             97
6                                Terraria   9.99             97
264                                 OMORI  19.99             97
768  Slay the Princess — The Pristine Cut  13.49             97
948                      Finding Paradise   9.99             97
16                        Baldur's Gate 3  59.99             96
430                    Tabletop Simulator  19.99             96
960                      Streets of Rogue  19.99             96
603             Unheard - Voices of Crime   6.99            

In [38]:
#Return the lowest price ever (in dollars) for a game, and print None if CheapShark can't find it.


def get_lowest_price(appid):
    url = "https://www.cheapshark.com/api/1.0/games"
    response = requests.get(url, params={"steamAppID": appid})
    if response.status_code != 200:
        return None
    data = response.json()
    if len(data) == 0:
        return None
    return float(data[0]["cheapest"])

# test it on the first 3 matching games
test = runs_on_mac.head(15)
for i in range(len(test)):
    row = test.iloc[i]
    print({row["name"]}, {row["price"]}, {get_lowest_price(row["appid"])})

{'Stardew Valley'} {np.float64(14.99)} {None}
{'Hades'} {np.float64(24.99)} {None}
{'Vampire Survivors'} {np.float64(4.99)} {None}
{"Touhou Mystia's Izakaya"} {np.float64(5.99)} {None}
{'Mount & Blade: Warband'} {np.float64(19.99)} {None}
{'DAVE THE DIVER'} {np.float64(19.99)} {None}
{'Terraria'} {np.float64(9.99)} {None}
{'OMORI'} {np.float64(19.99)} {None}
{'Slay the Princess — The Pristine Cut'} {np.float64(13.49)} {None}
{'Finding Paradise'} {np.float64(9.99)} {None}
{"Baldur's Gate 3"} {np.float64(59.99)} {None}
{'Tabletop Simulator'} {np.float64(19.99)} {None}
{'Streets of Rogue'} {np.float64(19.99)} {None}
{'Unheard - Voices of Crime'} {np.float64(6.99)} {None}
{'Castle Crashers®'} {np.float64(14.99)} {None}
